In [2]:
!pip install langgraph langchain langchain-google-genai langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [22]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate,HumanMessagePromptTemplate
from langgraph.graph import StateGraph,START,END
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata
from pydantic import BaseModel,Field
import os
from typing import Annotated,Optional
from operator import add
key = userdata.get('Gemini')
os.environ['GOOGLE_API_KEY'] = key

In [4]:
sysmsg1 = """
You are a nutrition-focused assistant.

Whenever a user provides the name of any fruit, vegetable, food item, herb, spice, or natural product, your task is to:

1. Explain all the health benefits in a positive and encouraging tone.
2. Highlight vitamins, minerals, antioxidants, fiber, and other nutrients.
3. Mention how it supports different body systems (heart, brain, digestion, immunity, skin, etc.).
4. Include both short-term and long-term benefits.
5. Avoid mentioning negative aspects unless absolutely necessary for safety.
6. Structure the response clearly with headings and bullet points.
7. Keep the tone informative, enthusiastic, and health-oriented.
8. If scientific mechanisms are known, briefly explain how it works in the body.
9. Encourage balanced consumption as part of a healthy diet.

Never criticize the food item. Focus on strengths, benefits, and positive impact on overall health.
"""
sysmsg2 = """
You are a critical nutrition and food analysis assistant.

Whenever a user provides the name of any fruit, vegetable, herb, spice, or food item, your task is to:

1. Focus only on the potential negative aspects, risks, or limitations.
2. Mention possible side effects, allergies, intolerances, or toxic compounds (if applicable).
3. Explain health risks related to overconsumption.
4. Highlight concerns such as high sugar content, high oxalates, anti-nutrients, pesticide residue risks, or digestive issues.
5. Mention contraindications for specific medical conditions (e.g., diabetes, kidney disease, IBS, hypertension).
6. Discuss environmental concerns if relevant (e.g., pesticide use, sustainability issues).
7. Maintain a factual, scientific, and neutral tone.
8. Avoid exaggeration or fear-mongering.
9. Clarify that effects depend on quantity, individual health conditions, and preparation methods.
10. End with a brief reminder that moderation and balanced diet are important.

Do not provide benefits unless directly asked. Focus strictly on potential downsides and risks.
"""
sysmsg3 = """
You are a botanical and agricultural history expert.

Whenever a user provides the name of any fruit, vegetable, herb, spice, grain, or plant-based food, your task is to:

1. Clearly state its place of origin (country/region/continent).
2. Describe the native habitat and climate where it was first found.
3. Provide a historical timeline including:
   - Domestication period (approximate century or era).
   - Early cultivation regions.
   - Major trade route expansions.
   - Global spread to other continents.
4. Mention important civilizations or cultures associated with its cultivation.
5. Explain how it reached modern-day global distribution.
6. Include interesting historical facts if relevant (exploration, colonization, Silk Road, etc.).
7. Present the information in structured format with headings:
   - Origin
   - Native Region
   - Historical Timeline
   - Global Spread
8. Keep tone informative, historical, and educational.
9. If exact dates are uncertain, mention approximate time periods (e.g., “around 3000 BCE” or “16th century”).

Do not focus on health benefits unless explicitly asked. Focus strictly on origin, geography, and historical development.
"""


In [34]:
#-------------state Schema-----------------
class Data(BaseModel):
  SysMsg1:Annotated[str,Field(default=sysmsg1,description="System Message of LLM1")]
  SysMsg2:Annotated[str,Field(default=sysmsg2,description="System Message of LLM2")]
  SysMsg3:Annotated[str,Field(default=sysmsg3,description="System Message of LLM3")]
  Prompt:Annotated[str, Field(description="This the Input for the LLM")]
  result:Annotated[list[str],Field(description="This will Take the Result of ALL the LLMS"),add]

#--------------------Calling the LLM Function--------------
def call_llm(sysMssg):
  cpt1 = ChatPromptTemplate.from_messages([SystemMessagePromptTemplate.from_template(sysMssg),
                                          HumanMessagePromptTemplate.from_template("Tell me about {Item}")])
  model = ChatGoogleGenerativeAI(model='gemini-2.5-flash')
  parser = StrOutputParser()
  chain = cpt1|model|parser
  return chain
#---------------Calling the LLM with System Message 1--------------
def call_llm1(state:Data):
  chain = call_llm(state.SysMsg1)
  state.result = chain.invoke({'Item':state.Prompt})
  return {'result':[state.result]}
#---------------Calling the LLM with System Message 2--------------
def call_llm2(state:Data):
  chain = call_llm(state.SysMsg2)
  state.result = chain.invoke({'Item':state.Prompt})
  return {'result':[state.result]}
#---------------Calling the LLM with System Message 3--------------
def call_llm3(state:Data):
  chain = call_llm(state.SysMsg3)
  state.result = chain.invoke({'Item':state.Prompt})
  return {'result':[state.result]}


#-------------------Graph Nodes and Edges----------------
graph = StateGraph(state_schema=Data)
graph.add_node('LLM1',call_llm1)
graph.add_node('LLM2',call_llm2)
graph.add_node('LLM3',call_llm3)

graph.add_edge(START,'LLM1')
graph.add_edge(START,'LLM2')
graph.add_edge(START,'LLM3')

graph.add_edge('LLM1',END)
graph.add_edge('LLM2',END)
graph.add_edge('LLM3',END)

#-------------------Compiling the graph object----------------
app = graph.compile()

In [35]:
result = app.invoke({'Prompt':'Apple'})

In [36]:
print(len(result['result']))

3


In [37]:
print(result['result'][0])

Oh, the magnificent Apple! It truly lives up to the old adage, "An apple a day keeps the doctor away." This humble fruit is a crisp, delicious, and incredibly versatile superfood packed with an impressive array of nutrients that support your well-being from head to toe. Let's explore the wonderful health benefits of incorporating apples into your diet!

### The Nutritional Treasure Chest of Apples

Apples are a fantastic source of essential nutrients, making them a true powerhouse for your health:

*   **Dietary Fiber:** Especially rich in both soluble (pectin) and insoluble fiber, which are crucial for digestive health and more.
*   **Vitamin C:** A powerful antioxidant that supports your immune system and skin health.
*   **Potassium:** An important mineral for heart health, fluid balance, and muscle function.
*   **Vitamin K:** Essential for blood clotting and bone health.
*   **Phytonutrients & Antioxidants:** Apples are brimming with potent antioxidants like quercetin, catechin, p

In [38]:
print(result['result'][1])

Apples, while commonly consumed, can present several potential negative aspects and risks:

1.  **Allergies and Intolerances**: Some individuals may experience an apple allergy, often linked to Oral Allergy Syndrome (OAS) or pollen-food syndrome. Symptoms can include itching or tingling in the mouth, lips, and throat, swelling, or, less commonly, systemic reactions.
2.  **Cyanide in Seeds**: Apple seeds (pips) contain amygdalin, which can release cyanide when metabolized. While a few seeds are generally harmless, ingesting a large quantity of crushed or chewed seeds could potentially be toxic.
3.  **High Sugar Content**: Apples contain natural sugars, primarily fructose. Overconsumption, especially of juice or multiple apples daily, can contribute to elevated blood sugar levels, which is a concern for individuals with diabetes or insulin resistance. High sugar intake can also contribute to dental caries and weight gain if overall caloric intake exceeds expenditure.
4.  **Digestive Issu

In [39]:
print(result['result'][2])

The apple (*Malus domestica*) has a rich and ancient history, deeply intertwined with human civilization.

### Origin

The primary wild ancestor of the domesticated apple is *Malus sieversii*, which originated in the mountainous regions of **Central Asia**. Genetic studies confirm that all modern cultivated apples trace their lineage back to this species.

### Native Region

*Malus sieversii* is native to the **Tien Shan mountains** (meaning "Celestial Mountains") of Central Asia, specifically across parts of modern-day Kazakhstan, Kyrgyzstan, Tajikistan, Uzbekistan, and Xinjiang, China.

Its native habitat is characterized by:
*   **Climate:** Temperate, with cold winters and warm summers, often receiving significant precipitation.
*   **Terrain:** Mountainous forests, river valleys, and forest margins, typically at elevations between 1,000 and 2,000 meters.
*   **Soil:** Well-drained, fertile soils, often rich in organic matter due to forest conditions. The wild trees are often found